In [ ]:
%load_ext autoreload
%autoreload 2
import os
import re 
import sys
import numpy as np
import pandas as pd
import xarray as xr
from tqdm.notebook import tqdm
import plotly.graph_objects as go
from os.path import join as pjoin
from sklearn.metrics import mutual_info_score
from sklearn.linear_model import LinearRegression
from scipy.stats import pearsonr, spearmanr, zscore, kendalltau
from natsort import natsorted
import statsmodels.api as sm
from statsmodels.formula.api import ols
from numpy.random import RandomState, SeedSequence, MT19937

sys.path.append('../../')
import circletrack_behavior as ctb
import circletrack_neural as ctn
import place_cells as pc
import plotting_functions as pf

In [ ]:
## Settings
project_folder = ['MultiCon_Imaging']
experiment_folders = ['MultiCon_Imaging5', 'MultiCon_Imaging6', 'MultiCon_Imaging7']
dpath = f'../../../{project_folder[0]}'
fig_path = f'../../../Manuscripts/MultiCon/intermediate_plots/reward_distance'
int_data = f'../../../Manuscripts/MultiCon/intermediate_plots/intermediate_data'
chance_color = '#7d7d7d'
avg_color = '#287347'
subject_color = '#7d7d7d'
ce_colors = ['#7A22BC', '#378616']
ce_colors_dict = {'Two-context': '#378616', 'Multi-context': '#7A22BC'}
symbol_dict = {'Two-context': 'x', 'Multi-context': 'circle'}
symbols_list = ['x', 'circle']
context_colors = {'A': '#a9a9a9', 'B': '#dc267f', 'C': '#648fff', 'D': '#fe6100',
                  'A1-5': '#a9a9a9', 'A5-10': '#dc267f', 'A10-15': '#648fff'}
mouse_colors = ['midnightblue', 'darkred', 'darkorchid', 'darkturquoise']
session_list = [f'A{x}' for x in np.arange(1, 6)] + [f'B{x}' for x in np.arange(1, 6)] + [f'C{x}' for x in np.arange(1, 6)] + [f'D{x}' for x in np.arange(1, 6)]
control_list = [f'A{x}' for x in np.arange(1, 16)] + [f'B{x}' for x in np.arange(1, 6)]
day_list = [f'Day {x}' for x in np.arange(1, 21)]
bin_size = 0.06 ## 0.06 radians linear position equivalent to 2cm-wide bins
reward_bin_size = 0.09
pos_distances = np.arange(0, (reward_bin_size*4) + reward_bin_size, reward_bin_size) ## distance from reward locations
all_midpoints = np.arange(-(reward_bin_size * 3), (reward_bin_size * 3) + reward_bin_size, reward_bin_size)
time_bin_size = 1 ## in seconds
velocity_thresh = 10
centroid_distance = 4
data_of_interest = 'place_cells' ## one of behav, aligned_minian, aligned_place_cells, lin_behav
data_type = 'S'
conversion = 2 / 0.06 ## 2cm per 0.06 radians

if not os.path.exists(fig_path):
    os.makedirs(fig_path)

xr.set_options(keep_attrs=True)

rs = RandomState(MT19937(SeedSequence(24601)))

In [ ]:
## Set mouse information
## mc54 and mc51 are good example mice for Two-context and Multi-context on day 16
experiment = 'MultiCon_Imaging5'
mouse = 'mc51'
day_of_int = '16'
session = f'{mouse}_{data_type}_{day_of_int}.nc'
only_running = True
correct_dir = True
cell_type = 'all_cells'

In [ ]:
## Load and process data
exp_path = pjoin(dpath, f'{experiment}/output/eli_test/{data_of_interest}/')
mpath = pjoin(exp_path, f'{mouse}/{data_type}')

sdata = xr.open_dataset(pjoin(mpath, session))[data_type] ## don't need to perform ctn.qc_matrix because aligned_place_cells already was qced
sdata = sdata[sdata['minimum_activity_met'], :] ## only use cells that meet activity threshold
if cell_type == 'place_cells':
    sdata = sdata[sdata['skaggs_place'], :] ## only select cells previously defined as place cells
elif cell_type == 'nonplace_cells':
    sdata = sdata[~sdata['skaggs_place'], :]
else:
    pass
spatial_info = sdata['skaggs_info'].values ## get an array of spatial info values
neural_data, position_data = ctn.subset_correct_dir_and_running(sdata, correct_dir=correct_dir, only_running=only_running, 
                                                                velocity_thresh=velocity_thresh)
## Get spike counts across spatial bins (population_activity) and rate maps (tuning_curves)
population_activity, occupancy, bins = pc.spatial_activity(neural_data, position_data, bin_size=bin_size, fps=30)
active_cells = np.sum(population_activity, axis=0) != 0
population_activity = population_activity[:, active_cells] ## remove any cells that have no activity after binning
tuning_curves = pc.get_tuning_curves(population_activity, occupancy)
tuning_curves = tuning_curves.T ## cells x spatial bin

## Find the peak of each place field
pf_peaks = np.max(tuning_curves, axis=1)
## Find the spatial bins where each peak occurred
field_dist = np.zeros(pf_peaks.shape[0])
for idx, peak in enumerate(pf_peaks):
    field_dist[idx] = bins[np.where(tuning_curves[idx, :] == peak)[0][0]]

## Get reward positions
reward_one_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_one']) & (sdata['correct_dir'])].values)
reward_two_pos = np.mean(sdata['lin_position'][(sdata['lick_port'] == sdata.attrs['reward_two']) & (sdata['correct_dir'])].values)

## Make Reward 1 the first rewarding port the mouse got water from
first_rew = sdata['lick_port'][sdata['water']].values[0]
if first_rew == sdata.attrs['reward_one']:
    first_rw_pos = reward_one_pos 
    second_rw_pos = reward_two_pos
else:
    first_rw_pos = reward_two_pos 
    second_rw_pos = reward_one_pos

pos_vals = []
for start in pos_distances:
    rw_one_amount = np.sum((abs(field_dist - first_rw_pos) >= start) & (abs(field_dist - first_rw_pos) < start + reward_bin_size))
    rw_two_amount = np.sum((abs(field_dist - second_rw_pos) >= start) & (abs(field_dist - second_rw_pos) < start + reward_bin_size))
    pos_vals.append((rw_one_amount + rw_two_amount) / tuning_curves.shape[0])

all_vals = []
for pos in all_midpoints:
    bin_start, bin_end = pos - (reward_bin_size / 2), pos + (reward_bin_size / 2)
    rw_one_amount = np.sum(((field_dist - first_rw_pos) >= bin_start) & ((field_dist - first_rw_pos) < bin_end))
    rw_two_amount = np.sum(((field_dist - second_rw_pos) >= bin_start) & ((field_dist - second_rw_pos) < bin_end))
    all_vals.append((rw_one_amount + rw_two_amount) / tuning_curves.shape[0])

In [ ]:
## Example proportion of place fields for all distances centered at x-axis values
fig = pf.custom_graph_template(x_title='Reward Distance (cm)', y_title='Proportion Place Fields')
fig.add_trace(go.Scattergl(x=all_midpoints * conversion, y=all_vals, mode='lines+markers', marker_size=8))
fig.update_yaxes(range=[0, np.max(all_vals) + 0.01])
fig.show()

In [ ]:
## Example distribution of place fields across track
rw_bins = np.arange(0, 6.28 + reward_bin_size, reward_bin_size)
H, xbin = np.histogram(field_dist, bins=rw_bins)
fig = pf.custom_graph_template(x_title='Spatial Bin (rad)', y_title='Proportion Place Fields', width=600)
hnorm = H / np.sum(H) ## convert to proportion
fig.add_trace(go.Bar(x=xbin, y=hnorm, marker_color=ce_colors_dict['Multi-context'], marker_line_width=2, marker_line_color='black'))
for val in [reward_one_pos, reward_two_pos]:
    fig.add_vline(x=val, line_dash='dash', line_width=3, opacity=0.7, line_color='red')
fig.update_yaxes(range=[0, 0.05])
fig.show()